<a href="https://colab.research.google.com/github/YangliuF95/text-as-data-exercises/blob/main/textual_analysis_with_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Textual analysis with Python: Beginner Friendly powered by [Huggingface](https://huggingface.co/) 🤗

**Goal:** This notebook introduces core textual-analysis methods using small toy examples. **It is designed for students with little or no previous Python experience. Please carefully go through this notebook line by line and run all the cells.**

By the end of this notebook, you should be able to:

1. Run cells in Google Colab or Jupyter Notebook.
2. Store and inspect short texts in Python.
3. Create simple word counts and a document-feature matrix (DFM).
4. Use a small dictionary to count concepts in texts.
5. Calculate TF-IDF scores.
6. Compare texts using sentence embeddings.
7. Run a simple topic model.
8. Use off-the-shelf transformer pipelines for sentiment and zero-shot classification.
9. Generate text with a small language model and reflect on limitations.
10. Understand why validation and careful interpretation are necessary.

---

## ❗Before you start: How to use this notebook

- Run a cell with **Shift + Enter**.
- Add your answer to each task by double-clicking the Task cell and type your answer after "**Your answer:**"
- If something goes wrong, choose **Runtime → Restart runtime** and then run the cells again from the top.
- Code comments begin with `#`. They explain what the code is doing.
- **Please use AI coding tutors, such as Toggle Gemini at the bottom of the page ❇️**
- ✴️You are not expected to memorize Python syntax. Focus on what the method does and how to interpret the results.✴️

## 0. Environment setup

The next cell installs and imports the packages we need.

In Google Colab, installation can take a few minutes. If you already have these packages installed, the cell may finish quickly.

In [ ]:
# Install packages used in this notebook.
# In Colab, run this cell once at the beginning.
!pip -q install pandas scikit-learn matplotlib sentence-transformers transformers torch

In [ ]:
# Import standard packages
import re
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make pandas output easier to read
pd.set_option("display.max_colwidth", 120)

## 1. A tiny toy corpus

A **corpus** is a collection of texts. Here we use short toy examples so that the output is easy to inspect.

The examples contain three rough themes:

- a. political/media-related statements
- b. coffee-related statements
- c. climate-policy statements

This makes it easier to see how different methods represent textual similarity and topics.

In [ ]:
texts = [ # this is the the corpus
    "The corrupt media protects powerful elites.",
    "Journalists hide scandals involving politicians and billionaires.",
    "Fresh coffee tastes rich and smooth.",
    "I enjoy drinking warm coffee in the morning.",
    "The government announced a new climate policy.",
    "Citizens debate whether the climate policy is fair."
]

labels = [ # these are the topics corresponding to each text in the corpus, the first one "the corrupt media xxx" corresponds to "politics_media", etc
    "politics_media",
    "politics_media",
    "coffee",
    "coffee",
    "climate_policy",
    "climate_policy"
]

# this is a pandas 🐼 dataframe that stores the texts and the labels
df = pd.DataFrame({"text": texts, "label": labels})
df

### Task 1

Add one more sentence to the `texts` list and one corresponding label to the `labels` list. Then rerun the cell above. (💡hint: always ask Gemini whenever there is an error)

For example, add another climate-policy sentence or another coffee sentence.

## 2. Python basics for text

Text is stored as a **string**. A collection of texts can be stored as a **list**. We can loop through a list and apply the same operation to each text.

In [ ]:
example_text = "The corrupt media protects powerful elites." # you can change the example

print(example_text) # the original text
print(example_text.lower())          # lowercase the text such that "The" becomes "the"
print(example_text.split())          # split into words using spaces such that a text will become a list of words

In [ ]:
# this helps us to print all the texts in the corpus
for text in texts:
    print(text.lower())

## 3. Preprocessing: lowercase, remove punctuation, tokenize

Before counting words, we usually apply simple preprocessing.

Here we use a small tokenizer function:

1. lowercase the text
2. remove punctuation
3. split the text into words

In [ ]:
# this is a function that helps us to tokenize the text
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # keep letters and spaces only
    tokens = text.split()
    return tokens

for text in texts[:2]: # [:2] means only focusing on the first 2 texts
    print(text)
    print(simple_tokenize(text))
    print()

### Task 2

Try the tokenizer on this sentence:

`"Democracy, climate, and media are important topics!"`

What changes after preprocessing?
**Your answer:**

In [ ]:
# run this code
simple_tokenize("Democracy, climate, and media are important topics!")

## 4. Counting words ⭐

A simple starting point in textual analysis is to ask:

> Which words appear most often?

This is useful, but it ignores grammar, word order, context, sentiment, and meaning.

In [ ]:
all_tokens = [] #put all the words from the corpus together for counting

for text in texts:
    all_tokens.extend(simple_tokenize(text))

word_counts = Counter(all_tokens) # the output is (word, frequency)
word_counts.most_common(15) # print the most common 15 words based on the counting

In [ ]:
# creat a 🐼 dataframe to store the data structurally
word_count_df = pd.DataFrame(word_counts.most_common(15), columns=["word", "count"])
word_count_df

In [ ]:
# visualize the data, the x axis shows the words and the y axis shows the count
plt.figure(figsize=(8, 4))
plt.bar(word_count_df["word"], word_count_df["count"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Word")
plt.ylabel("Count")
plt.title("Most frequent words in the toy corpus")
plt.show()

### Task 3

Look at the most frequent words. Which words are meaningful? Which words are less informative?

Write 2–3 sentences below.

**Your answer:**

## 5. Document-feature matrix (DFM) ⭐

A **document-feature matrix** represents documents as rows (horizontal) and words/features as columns (vertical).

Each cell records how often a word appears in a document.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer # use established python packages

count_vectorizer = CountVectorizer(lowercase=True, stop_words="english") # vectorize the counting
X_counts = count_vectorizer.fit_transform(texts)

terms = count_vectorizer.get_feature_names_out()

dfm = pd.DataFrame(X_counts.toarray(), columns=terms) # construct the dfm
dfm.insert(0, "text", texts) # add the original texts to the dfm
dfm # show dfm

### Task 4

Find the column for `coffee`, `policy`, or `climate`. Which documents contain those words?
**Your answer:**

## 6. Dictionary analysis ⭐

A **dictionary** groups words into categories. For example, we can define a small toy dictionary for political concepts.

This is simple and transparent, but it depends heavily on which words we include.

In [ ]:
dictionary = { # predefine a dictionary manually
    "media_elite": ["media", "journalists", "elites", "billionaires"],
    "politics": ["government", "policy", "politicians", "citizens", "debate"],
    "climate": ["climate"],
    "coffee": ["coffee", "drinking", "morning", "smooth"],
}

def count_dictionary_categories(text, dictionary): # the function that count the text based on the predefined the dictionary categories
    tokens = simple_tokenize(text)
    counts = {}
    for category, words in dictionary.items():
        counts[category] = sum(token in words for token in tokens)
    return counts

dict_results = [] # transform the texts using the dictionaries such that we could reduce the vocabulary
for text in texts:
    row = {"text": text}
    row.update(count_dictionary_categories(text, dictionary))
    dict_results.append(row)

pd.DataFrame(dict_results)

### Task 5

Add a new category to the dictionary called `emotion` with words such as `fair`, `corrupt`, `enjoy`, or `powerful`. Rerun the dictionary analysis.

Does the new category capture what you expected?
**Your answer:**

## 7. TF-IDF ⭐

**TF-IDF** stands for **term frequency–inverse document frequency**.

It gives higher scores to unique words that are frequent in one document but not common across all documents.

This helps identify words that are distinctive for particular documents.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer #use established packages

tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(texts) # transform our texts

tfidf_terms = tfidf_vectorizer.get_feature_names_out() # get the words
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_terms)  # get the tf-idf scores for each word in each document
tfidf_df.insert(0, "text", texts)
tfidf_df

In [ ]:
# Show the top TF-IDF terms for each document
for i, text in enumerate(texts):
    scores = tfidf_df.drop(columns="text").iloc[i]
    top_terms = scores.sort_values(ascending=False).head(5)
    print(f"Document {i}: {text}")
    print(top_terms)
    print()

### Task 6

Choose a document and inspect which terms receive the highest TF-IDF scores. Do they seem distinctive for that document? **Your answer:**

## 8. Embeddings: representing meaning as vectors ⭐

Counting methods represent texts using words. Embedding models try to represent the **meaning** of texts as numerical vectors. See my example of scientific text [method2vec](https://yangliuf95.github.io/).

Here we use a small sentence-transformer 🤖 model. It turns each sentence into a vector. We then calculate **cosine similarity** between sentence vectors.

- The higher the cosine similarity, the more similar the texts are. A cosine similarity closer to 1 means the texts are almost identical.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_sentences = [
    "I feel great this morning.",
    "I am feeling wonderful today.",
    "The government announced a new climate policy.",
    "Parliament debated environmental regulation.",
    "Fresh coffee tastes rich and smooth."
]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2") # a small pretrained model
embeddings = embedding_model.encode(embedding_sentences)

similarity_matrix = cosine_similarity(embeddings)

pd.DataFrame(similarity_matrix, index=embedding_sentences, columns=embedding_sentences) # the cells with numbers indicate the cosine similarity between two texts
# you need ingore the diagonal between those are the self similarity scores (i.e., 1)

### Task 7

Which two sentences are most similar? Does the result make sense?

Then add a new sentence that is a synonym-like version of one sentence, for example:

`"I feel fantastic this morning."`

Rerun the cell and inspect the similarity matrix again.

## 9. Topic modeling with LDA ⭐

Topic modeling is an unsupervised method for discovering clusters of words that tend to appear together.

❗❗❗Important: With only a few toy texts, topic modeling is not meaningful. We use it here only to understand the logic of the method.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# We use a slightly larger toy corpus to make the topics easier to see.
topic_texts = [
    "The corrupt media protects powerful elites.",
    "Journalists hide scandals involving politicians and billionaires.",
    "The newspaper reported a scandal about political donors.",
    "Fresh coffee tastes rich and smooth.",
    "I enjoy drinking warm coffee in the morning.",
    "The cafe serves espresso and fresh pastries.",
    "The government announced a new climate policy.",
    "Citizens debate whether the climate policy is fair.",
    "Environmental regulation and climate targets shape political debate."
]

vectorizer = CountVectorizer(stop_words="english")
X_topics = vectorizer.fit_transform(topic_texts)
terms = vectorizer.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=3, random_state=42) # here you need to specify the number of topics in the n_component = x
lda.fit(X_topics)

for topic_idx, topic in enumerate(lda.components_):
    top_word_indices = topic.argsort()[-6:][::-1]
    top_words = [terms[i] for i in top_word_indices]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

In [ ]:
# Document-topic mixtures
doc_topic = lda.transform(X_topics)
doc_topic_df = pd.DataFrame(doc_topic, columns=["Topic 1", "Topic 2", "Topic 3"]) # after you change the number of topics, you also need to change the columns here
doc_topic_df.insert(0, "text", topic_texts)
doc_topic_df # cells with numbers indicate the probability that the text contains a specific topic,
# you need to look at the output here in the context of the topics from the output above

### Task 8

Try changing `n_components=3` to `n_components=2` or `n_components=4`.

How do the topics change? What does this tell you about topic modeling?
**Your answer:**

## 10. Off-the-shelf sentiment analysis ⭐

Some NLP tasks can be done with pretrained models. These are sometimes called **off-the-shelf detectors**.

Here we use a sentiment-analysis pipeline.

Important: The model returns a label and a score, but that does not mean it is always correct. Models can fail when the domain, language, irony, or political context differs from the model's training data.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis") # a small sentiment analysis example

sentiment_texts = [
    "I love this policy.",
    "This debate was terrible.",
    "The government announced a new law.",
    "The corrupt media protects powerful elites.",
    "Citizens are worried about the new reform."
]

sentiment(sentiment_texts)

In [ ]:
sentiment_results = sentiment(sentiment_texts)

pd.DataFrame({
    "text": sentiment_texts,
    "label": [r["label"] for r in sentiment_results],
    "score": [r["score"] for r in sentiment_results]
})

### Task 9

Which result is most convincing? Which result is most questionable?

Explain why in 2–3 sentences.**Your answer:**

## 11. Zero-shot classification ⭐

Zero-shot classification assigns texts to categories that we define, even if the model was not trained specifically on our exact dataset.

This is useful for exploration, but it should be validated before being used as evidence in research.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = ["economy", "climate", "migration", "democracy", "foreign policy", "media"] # this is the classification labels we want

statement = "The government announced new climate targets and environmental regulation." # an example

zero_shot(statement, candidate_labels) # the label with the highest score is one the model assigned

In [ ]:
# this takes some time to classify
statements = [
    "The government announced new climate targets and environmental regulation.",
    "The newspaper reported scandals involving powerful politicians.",
    "Parties disagree about border control and asylum policy.",
    "Citizens protested against threats to democratic institutions."
]

rows = []
for statement in statements:
    result = zero_shot(statement, candidate_labels)
    rows.append({
        "text": statement,
        "top_label": result["labels"][0],
        "top_score": result["scores"][0]
    })

pd.DataFrame(rows)

### Task 10

Add one new political statement and classify it using the zero-shot model.

Is the top label reasonable? Would you trust it in a research project without validation? **Your answer:**

## 12. Finally: text generation with a small language model ⭐

Language models generate text token by token. Their output is probabilistic: the same prompt may produce different continuations.

We use a small model here so the example runs quickly.

In [ ]:
# this takes a long time to run
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Political polarization is" # we want the model to generate some text about political polarization

outputs = generator(
    prompt,
    max_length=40,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.9,
    pad_token_id=50256
)

for i, output in enumerate(outputs, start=1):
    print(f"Output {i}:")
    print(output["generated_text"])
    print()

### Task 11

Run the generation cell two or three times.

Do you get the same output every time? What does this suggest about using LLM in research? **Your answer:**

## 13. Comparing methods

Use this table to summarize what each method gives you.

In [ ]:
method_summary = pd.DataFrame({
    "method": [
        "Counting / DFM",
        "Dictionary analysis",
        "TF-IDF",
        "Embeddings",
        "Topic modeling",
        "Off-the-shelf classification",
        "Zero shot classification",
        "Text generation"
    ],
    "main_question": [
        "What words appear?",
        "How often do predefined concepts appear?",
        "Which words are distinctive?",
        "Which texts are semantically similar?",
        "What themes occur?",
        "Which predefined category does this text belong to?",
        "Can a pre-trained model without finetuning follow instructions to classify or extract information?",
        "What continuation can a language model produce?"
    ],
    "output": [
        "counts / frequencies",
        "category counts",
        "weighted word scores",
        "vectors / similarity scores",
        "topics and document-topic mixtures",
        "labels and scores",
        "structured labels, extracted fields, explanations",
        "generated text"
    ],
    "main_caution": [
        "Ignores context and word order",
        "Depends on dictionary design",
        "Still bag-of-words; limited meaning",
        "Harder to interpret directly",
        "Requires interpretation; sensitive to settings",
        "May not fit your domain or language",
        "May be inconsistent or biased; needs validation",
        "Can hallucinate; output is probabilistic"
    ]
})

method_summary

## Final reflection task

Write a short reflection on the following questions:

1. Which method was easiest to understand?
2. Which method seemed most useful for political text analysis?
3. Which method seemed most risky or easiest to misuse?
4. Why is validation important when using off-the-shelf models or LLMs?

**Your answer:**